# 09 — Deep Learning: MLP e BrainNetCNN

Este experimento avalia **dois modelos** sobre a mesma base validada de 871 participantes: um MLP recebe o vetor com 19.900 conexões; a BrainNetCNN recebe a matriz 200 × 200 e usa operações E2E (edge-to-edge), E2N (edge-to-node) e N2G (node-to-graph). Ambos usam validação cruzada externa agrupada por site, busca/seleção somente dentro do treino e previsões OOF. As estatísticas de padronização são ajustadas apenas no treino interno de cada fold.

A arquitetura BrainNetCNN aqui está implementada explicitamente neste projeto; não é uma CNN genérica renomeada.

Resultados e pesos são gravados em `Results/modelos_dl_tea/` e `Models/modelos_dl_tea/`, separados das métricas preservadas dos outros modelos.


In [1]:
# 1. Projeto, dados e reprodução
import sys, copy, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import torch
import torch.nn as nn

_cwd=Path.cwd().resolve()
_raiz=next((p for p in [_cwd,*_cwd.parents] if (p/'src'/'config.py').is_file()),None)
if _raiz is None: raise FileNotFoundError('Não encontrei src/config.py; abra o notebook dentro do projeto TCC_TEA.')
sys.path.insert(0,str(_raiz)) if str(_raiz) not in sys.path else None
from src import config, avaliacao as av
from src.modelos_dl import MLPConectividade, BrainNetCNN
config.criar_pastas()
SEED=int(config.SEED); ARQ_DATASET=_raiz/'Data'/'processed'/'dataset_abide_processado.pkl'
PASTA_RES=_raiz/'Results'/'modelos_dl_tea'; PASTA_MODELOS=_raiz/'Models'/'modelos_dl_tea'
PASTA_RES.mkdir(parents=True,exist_ok=True); PASTA_MODELOS.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
d=av.carregar_dataset(ARQ_DATASET)
N_ROIS=av.n_rois_de(d['X'].shape[1]); n=len(d['y'])
_i,_j=np.triu_indices(N_ROIS,k=1)
MATS=np.zeros((n,N_ROIS,N_ROIS),dtype=np.float32)
MATS[:,_i,_j]=d['X']; MATS[:,_j,_i]=d['X']
print(f'Dataset: {n} pessoas; X={d["X"].shape}; {N_ROIS} ROIs; {len(np.unique(d["sites"]))} sites; dispositivo={DEVICE}')
print('Classes:',pd.Series(d['y']).value_counts().sort_index().to_dict())

# 2. Configuração do experimento
SEMENTES=[42,43,44]; EPOCAS_FIXAS=20
EPOCAS_MLP, PACIENCIA_MLP = 40, 6
EPOCAS_BRAINNET, PACIENCIA_BRAINNET = 12, 3
BATCH_MLP=64; BATCH_BRAINNET=8
LR=1e-3; WD=1e-4
if av.RAPIDO:
    SEMENTES=[42]; EPOCAS_FIXAS=3
    EPOCAS_MLP, PACIENCIA_MLP = 3, 2
    EPOCAS_BRAINNET, PACIENCIA_BRAINNET = 3, 2
    print('*** TCC_RAPIDO=1: execução curta para inspeção do fluxo ***')

Dataset: 871 pessoas; X=(871, 19900); 200 ROIs; 20 sites; dispositivo=cpu
Classes: {0: 468, 1: 403}


In [2]:
# 3. Funções comuns de normalização, treino e previsão
def normalizar_matrizes(idx_treino):
    """Padroniza as matrizes com média/desvio escalares estimados só no treino."""
    off=~np.eye(N_ROIS,dtype=bool); valores=MATS[idx_treino][:,off]
    mu=float(valores.mean()); sd=float(valores.std()+1e-6)
    out=((MATS-mu)/sd).astype(np.float32)
    out[:,np.arange(N_ROIS),np.arange(N_ROIS)]=0
    return out,mu,sd

def normalizar_vetor(idx_treino):
    """Padroniza cada conexão usando participantes do treino externo apenas."""
    mu=d['X'][idx_treino].mean(0); sd=d['X'][idx_treino].std(0)+1e-6
    return ((d['X']-mu)/sd).astype(np.float32)

@torch.no_grad()
def prever(modelo,X,batch=32):
    modelo.eval(); out=[]
    for k in range(0,len(X),batch):
        xb=torch.as_tensor(X[k:k+batch],dtype=torch.float32,device=DEVICE)
        out.append(torch.sigmoid(modelo(xb)).detach().cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def treinar(modelo,Xtr,ytr,Xva=None,yva=None,semente=42,batch_size=BATCH_MLP,epocas_max=40,paciencia=6):
    """BCE ponderada; early stopping por AUC interna, ou épocas fixas se inviável."""
    modelo=modelo.to(DEVICE); gen=torch.Generator().manual_seed(semente)
    pos=max(int((ytr==1).sum()),1); neg=max(int((ytr==0).sum()),1)
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/pos],device=DEVICE))
    opt=torch.optim.AdamW(modelo.parameters(),lr=LR,weight_decay=WD)
    tem_val=Xva is not None and len(np.unique(yva))==2
    max_ep=epocas_max if tem_val else EPOCAS_FIXAS
    best=-1.; state=None; sem_melhora=0; hist=[]
    Xt=torch.as_tensor(Xtr,dtype=torch.float32); yt=torch.as_tensor(ytr,dtype=torch.float32)
    for ep in range(max_ep):
        modelo.train(); order=torch.randperm(len(Xt),generator=gen); soma=0.
        for k in range(0,len(order),batch_size):
            b=order[k:k+batch_size]
            xb=Xt[b].to(DEVICE); yb=yt[b].to(DEVICE)
            opt.zero_grad(set_to_none=True); loss=loss_fn(modelo(xb),yb); loss.backward(); opt.step()
            soma+=loss.item()*len(b)
        auc=np.nan
        if tem_val:
            auc=av.roc_auc_score(yva,prever(modelo,Xva,batch=batch_size))
            if auc>best+1e-4: best=auc; state=copy.deepcopy(modelo.state_dict()); sem_melhora=0
            else: sem_melhora+=1
        hist.append(soma/max(len(Xt),1))
        if tem_val and sem_melhora>=paciencia: break
    if tem_val and state is not None: modelo.load_state_dict(state)
    return modelo,{'auc_val_interna':best if tem_val else np.nan,'epocas':len(hist),'perda':hist}

def escolher_limiar(yva,pva):
    """Escolhe limiar pela acurácia balanceada da validação interna; padrão 0,5 sem validação."""
    if yva is None or len(np.unique(yva))<2: return 0.5
    cand=np.unique(np.round(pva,4))
    return float(max((av.balanced_accuracy_score(yva,(pva>=t).astype(int)),-abs(t-.5),t) for t in cand)[2])

print('Modelos:', MLPConectividade.__name__, 'e', BrainNetCNN.__name__)

Modelos: MLPConectividade e BrainNetCNN


In [3]:
# 4. Validação cruzada externa agrupada por site
print('FOLDS EXTERNOS'); display(av.resumo_folds(d))
modelos=['MLP','BrainNetCNN']; folds=sorted(np.unique(d['fold_id']).tolist())
proba_oof={m:np.full(n,np.nan,dtype=np.float32) for m in modelos}
pred_oof={m:np.full(n,-1,dtype=np.int8) for m in modelos}; linhas=[]
for fold in folds:
    te=np.where(d['fold_id']==fold)[0]; tr=np.where(d['fold_id']!=fold)[0]
    splits,info=av.splits_internos(d['y'][tr],d['sites'][tr],k_max=3,seed=SEED)
    in_tr,in_va=(tr[splits[0][0]],tr[splits[0][1]]) if info['viavel'] else (tr,None)
    X_mlp=normalizar_vetor(in_tr); X_bnc,_,_=normalizar_matrizes(in_tr)
    yva=None if in_va is None else d['y'][in_va]
    entradas={'MLP':X_mlp,'BrainNetCNN':X_bnc[:,None,:,:]}
    print(f'\nFold {fold}: treino={len(tr)}, teste={len(te)}, validação interna={0 if in_va is None else len(in_va)}')
    for nome in modelos:
        X=entradas[nome]
        for sem in SEMENTES:
            torch.manual_seed(sem*100+fold); np.random.seed(sem*100+fold); t0=av.agora()
            if nome=='MLP': modelo=MLPConectividade(X.shape[1])
            else: modelo=BrainNetCNN(N_ROIS)
            bs=BATCH_MLP if nome=='MLP' else BATCH_BRAINNET
            epmax=EPOCAS_MLP if nome=='MLP' else EPOCAS_BRAINNET
            pat=PACIENCIA_MLP if nome=='MLP' else PACIENCIA_BRAINNET
            modelo,info_t=treinar(modelo,X[in_tr],d['y'][in_tr],None if in_va is None else X[in_va],yva,sem,batch_size=bs,epocas_max=epmax,paciencia=pat)
            pva=None if in_va is None else prever(modelo,X[in_va],batch=bs); lim=escolher_limiar(yva,pva)
            p=prever(modelo,X[te],batch=bs); pr=(p>=lim).astype(np.int8)
            linha=av.linha_resultado('TEA',nome,fold,sem,d['y'][te],pr,p,limiar=lim,
                auc_val_interna=info_t['auc_val_interna'],epocas=info_t['epocas'],
                k_interno=int(info['k_usado']),seg_total=round(av.agora()-t0,1))
            linhas.append(linha)
            if sem==SEMENTES[0]:
                proba_oof[nome][te]=p; pred_oof[nome][te]=pr
                pasta=PASTA_MODELOS/nome.lower(); pasta.mkdir(parents=True,exist_ok=True)
                torch.save(modelo.state_dict(),pasta/f'{nome.lower()}_fold{fold}.pt')
            alerta=' | ALERTA: previu uma classe só' if linha['classe_unica_prevista'] else ''
            print(f" {nome:12s} seed={sem} AUC={linha['auc_roc']:.3f} BalAcc={linha['balanced_accuracy']:.3f} limiar={lim:.3f} épocas={info_t['epocas']}{alerta}")
        pd.DataFrame(linhas).to_csv(PASTA_RES/'09_tea_resultados_long.csv',index=False)
for nome in modelos:
    if np.any(pred_oof[nome]<0) or not np.isfinite(proba_oof[nome]).all(): raise RuntimeError(f'Previsões OOF incompletas para {nome}')
df_long=pd.DataFrame(linhas); df_long['tipo']='externo' 

FOLDS EXTERNOS


,fold,n_treino,n_teste,controles_teste,casos_teste,controles_treino,casos_treino,sites_teste,sobreposicao_sites,duas_classes_no_teste
0,0,830,41,19,22,449,381,YALE,0,True
1,1,698,173,101,72,367,331,"KKI, OLIN, SBL, UM_1",0,True
2,2,706,165,91,74,377,329,"CALTECH, LEUVEN_1, LEUVEN_2, PITT, TRINITY",0,True
3,3,547,324,175,149,293,254,"CMU, NYU, OHSU, SDSU, STANFORD, UCLA_1",0,True
4,4,703,168,82,86,386,317,"MAX_MUN, UCLA_2, UM_2, USM",0,True



Fold 0: treino=830, teste=41, validação interna=158
 MLP          seed=42 AUC=0.833 BalAcc=0.743 limiar=0.662 épocas=10
 MLP          seed=43 AUC=0.789 BalAcc=0.755 limiar=0.498 épocas=29
 MLP          seed=44 AUC=0.768 BalAcc=0.713 limiar=0.555 épocas=10
 BrainNetCNN  seed=42 AUC=0.711 BalAcc=0.687 limiar=0.246 épocas=11
 BrainNetCNN  seed=43 AUC=0.766 BalAcc=0.626 limiar=0.397 épocas=8
 BrainNetCNN  seed=44 AUC=0.782 BalAcc=0.732 limiar=0.596 épocas=7

Fold 1: treino=698, teste=173, validação interna=92
 MLP          seed=42 AUC=0.689 BalAcc=0.561 limiar=0.298 épocas=10
 MLP          seed=43 AUC=0.745 BalAcc=0.565 limiar=0.124 épocas=15
 MLP          seed=44 AUC=0.750 BalAcc=0.658 limiar=0.207 épocas=20
 BrainNetCNN  seed=42 AUC=0.657 BalAcc=0.607 limiar=0.095 épocas=6
 BrainNetCNN  seed=43 AUC=0.646 BalAcc=0.592 limiar=0.360 épocas=6
 BrainNetCNN  seed=44 AUC=0.623 BalAcc=0.513 limiar=0.524 épocas=4

Fold 2: treino=706, teste=165, validação interna=178
 MLP          seed=42 AUC=0.5

In [4]:
# 5. Resumo, intervalos e arquivos de saída
res=av.resumir_resultados(df_long,d,proba_oof)
print('MÉDIA ± DP ENTRE FOLDS'); display(res['por_fold'])
print('AUC OOF e IC 95% por bootstrap de sites'); display(res['auc_oof_ic'])
print('A interpretação de “acima do acaso” exige limite inferior do IC da AUC > 0,5.')
print('Execuções que previram uma única classe:',int(df_long['classe_unica_prevista'].sum()),'de',len(df_long))
df_long.to_csv(PASTA_RES/'09_tea_resultados_long.csv',index=False)
res['por_fold'].to_csv(PASTA_RES/'09_tea_resumo_por_fold.csv')
res['ponderado'].to_csv(PASTA_RES/'09_tea_resumo_ponderado.csv',index=False)
res['auc_oof_ic'].to_csv(PASTA_RES/'09_tea_auc_oof_ic.csv',index=False)
av.salvar_oof(PASTA_RES/'09_tea_predicoes_oof.npz',d,proba_oof,pred_oof)
print('Resultados em:',PASTA_RES); print('Pesos por fold em:',PASTA_MODELOS)

MÉDIA ± DP ENTRE FOLDS


accuracy         balanced_accuracy         precision          \
                mean     std              mean     std      mean     std   
modelo                                                                     
BrainNetCNN   0.6220  0.0609            0.6281  0.0485    0.6321  0.1213   
MLP           0.6412  0.0714            0.6478  0.0623    0.6410  0.1226   

             recall         specificity              f1         auc_roc  \
               mean     std        mean     std    mean     std    mean   
modelo                                                                    
BrainNetCNN  0.6078  0.1120      0.6483  0.1934  0.5979  0.0408  0.6914   
MLP          0.6374  0.1832      0.6581  0.2117  0.6177  0.0970  0.7182   

                     
                std  
modelo               
BrainNetCNN  0.0513  
MLP          0.0530

AUC OOF e IC 95% por bootstrap de sites


,modelo,auc,ic_inf,ic_sup,n_boot_validos,obs
0,MLP,0.6867,0.6367,0.7243,2000,
1,BrainNetCNN,0.6850,0.6447,0.7156,2000,


A interpretação de “acima do acaso” exige limite inferior do IC da AUC > 0,5.
Execuções que previram uma única classe: 0 de 30
Resultados em: /home/maria/TCC_TEA/Results/modelos_dl_tea
Pesos por fold em: /home/maria/TCC_TEA/Models/modelos_dl_tea
